# EnderLeaf script preparation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

## Imports

In [ ]:
from pathlib import Path
import time

from tqdm import tqdm
from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

from enderleaf.image import (
    to_pil,
    safe_pil_resize,
    merge_images,
    ImageMergeMode,
    get_circles,
    crop_image,
    Rectangle,
)
from enderscope.bed import bed
from enderscope.scan_patterns import plot_path_status
from enderleaf.focus_metrics import compute_focus_metric, FM_METHODS, FM_BREN
from enderleaf.draw import image_grid, concat_tile_resize
from enderleaf.enderleaf_ui import ui_show, controller, ui_main, ui_sidebar
from enderleaf.enderleaf_ctrl import CameraState
from enderscope.enderlights_pi import CardPoint, LIGHTS_CYCLE, LEN_LIGHTS_CYCLE
from enderleaf.tools import ensure_folder, format_datetime, write_dataframe

import panel as pn

## Initialize Preview

In [ ]:
ui_show().servable()

In [ ]:
controller.camera.set_controls(
    {
        "AeEnable": False,
        "ExposureTime": 4000,
        "AnalogueGain": 1,
        "AwbEnable": False,
        "ColourGains": (2.05, 1.05),
    }
)

In [ ]:
controller.set_focus_close()

In [ ]:
controller.focus_methods = FM_METHODS
controller.get_focused_z(switch_state=True)

In [ ]:
controller.center_on_qr_code(switch_state=True, precise_focusing=False)

In [ ]:
controller.focus_methods = FM_METHODS
data = {"idx": [], "light": [], "combined": []} | {fm: [] for fm in FM_METHODS}
controller.switch_state(CameraState.STILL)
for light in [True, False]:
    controller.shutter(light)
    for i in range(2):
        result = controller.get_focused_z(switch_state=False)
        data["idx"].append(i)
        data["light"].append(light)
        for k, v in result.items():
            data[k].append(v)
controller.switch_state(CameraState.VIDEO)

df_fm = pd.DataFrame(data)
df_fm

In [ ]:
df_scores = controller.get_focused_z(switch_state=False)
df_scores

In [ ]:
pd.melt(df_scores, id_vars=["z"], value_vars=FM_METHODS + ["combined"])

In [ ]:
import plotly.express as px

px.line(
    pd.melt(df_scores, id_vars=["z"], value_vars=FM_METHODS + ["combined"]),
    x="z",
    y="value",
    color="variable",
    markers=True,
    width=400,
)

In [ ]:
df_groupped = (
    df_fm.groupby("light")
    .agg({k: ["mean", "std"] for k in FM_METHODS + ["combined"]})
    .reset_index()
)
pd.DataFrame(
    data={"light": df_groupped["light"]}
    | {
        k: [
            f'{df_groupped[k]["mean"][i]:.1f}±{df_groupped[k]["std"][i]:.2f}'.replace(
                "±nan", ""
            )
            for i in range(len(df_groupped))
        ]
        for k in FM_METHODS + ["combined"]
    }
).style

In [ ]:
controller.switch_state(CameraState.STILL)


controller.camera.set_controls(
    {
        "AeEnable": False,
        "ExposureTime": 500,
        "AnalogueGain": 1,
        "AwbEnable": False,
        "ColourGains": (2.05, 1.05),
    }
)
time.sleep(2)

images_shoff = [
    cv2.medianBlur(controller.capture_array()[0], ksize=5) for _ in tqdm(range(10))
]
focus_data_shoff = np.array(
    [
        compute_focus_metric(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY), FM_BREN)
        for image in tqdm(images_shoff)
    ]
)
img_min_shoff, img_max_shoff = (
    images_shoff[focus_data_shoff.argmin()],
    images_shoff[focus_data_shoff.argmax()],
)

controller.camera.set_controls(
    {
        "AeEnable": False,
        "ExposureTime": 8000,
        "AnalogueGain": 1,
        "AwbEnable": False,
        "ColourGains": (2.05, 1.05),
    }
)
time.sleep(2)


images_shon = [
    cv2.medianBlur(controller.capture_array()[0], ksize=7) for _ in tqdm(range(10))
]
focus_data_shon = np.array(
    [
        compute_focus_metric(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY), FM_BREN)
        for image in tqdm(images_shon)
    ]
)
img_min_shon, img_max_shon = (
    images_shon[focus_data_shon.argmin()],
    images_shon[focus_data_shoff.argmax()],
)

controller.switch_state(CameraState.VIDEO)

to_pil(
    concat_tile_resize(
        [
            [img_min_shoff, img_max_shoff, np.abs(img_max_shoff - img_min_shoff)],
            [img_min_shon, img_max_shon, np.abs(img_max_shon - img_min_shon)],
        ]
    )
)

In [ ]:
focus_data = np.array(
    [
        compute_focus_metric(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY), FM_BREN)
        for image in tqdm(images)
    ]
)
focus_data

In [ ]:
sel_img = pn.widgets.IntSlider(
    name="Image index", start=0, end=len(images), sizing_mode="scale_width"
)
img = pn.pane.Image(sizing_mode="scale_width")


@pn.depends(sel_img.param.value, watch=True)
def on_index_changed(index):
    img.object = to_pil(images[index])


on_index_changed(sel_img.value)

pn.Column(img, sel_img)

In [ ]:
focus_data.min(), focus_data.max(), focus_data.argmin(), focus_data.argmax()

In [ ]:
img_min, img_max = images[focus_data.argmin()], images[focus_data.argmax()]

to_pil(concat_tile_resize([[img_min, img_max]]))

In [ ]:
img_diff = 
img_diff

In [ ]:
controller.acq_light_configurations = [
    LIGHTS_CYCLE[2],
    LIGHTS_CYCLE[3],
    LIGHTS_CYCLE[4],
    LIGHTS_CYCLE[5],
]
controller.launch_acquisition(precise_focusing=True, switch_state=True, center_on_leaf=True)

In [ ]:
controller.switch_state(CameraState.STILL)
sel_pos = pn.widgets.IntSlider(
    name="Position",
    start=1,
    end=len(controller._positions),
    sizing_mode="stretch_width",
)
img = pn.pane.Image(sizing_mode="stretch_width")


@pn.depends(sel_pos.param.value, watch=True)
def on_pos_changed(pos):
    try:
        controller.move_to(pos)
    except:
        return
    image, _ = controller.capture_array()
    cur_data = {}
    cy, cx = image.shape[0] // 2, image.shape[1] // 2
    circles = get_circles(image, channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        a, ccx, ccy, r = circles["accepted"][0]
        cur_data["accu"] = a
        cur_data["cx"] = ccx
        cur_data["cy"] = ccy
        image = cv2.circle(image, (ccx, ccy), r, (0, 0, 0), 32)
        controller.center_on_target(cx, cy, ccx, ccy)
    moved_image, _ = controller.capture_array()
    circles = get_circles(moved_image, channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        a, ccx, ccy, r = circles["accepted"][0]
        cur_data["new_accu"] = a
        cur_data["new_cx"] = ccx
        cur_data["new_cy"] = ccy
    img.object = to_pil(concat_tile_resize([[circles["edges"], image, moved_image]]))


on_pos_changed(sel_pos.value)

pn.Column(sel_pos, img)

In [ ]:
controller.acq_light_configurations = [
    LIGHTS_CYCLE[2],
    LIGHTS_CYCLE[3],
    LIGHTS_CYCLE[4],
    LIGHTS_CYCLE[5],
]
controller.acq_light_configurations

In [ ]:
controller.launch_acquisition(precise_focusing=True, switch_state=True)

In [ ]:
plot_path_status(title="", z=10)

In [ ]:
controller.acq_light_configurations = [LIGHTS_CYCLE[0]]
pprint(controller.acq_light_configurations)
controller.set_lights(controller.acq_light_configurations[0], wait=1)
images_no_light, *_ = controller.acquire_leaf_disc(switch_state=True)

controller.acq_light_configurations = [LIGHTS_CYCLE[1]]
pprint(controller.acq_light_configurations)
controller.set_lights(controller.acq_light_configurations[0], wait=1)
images_full_light, *_ = controller.acquire_leaf_disc(switch_state=True)

controller.acq_light_configurations = [
    LIGHTS_CYCLE[2],
    LIGHTS_CYCLE[3],
    LIGHTS_CYCLE[4],
    LIGHTS_CYCLE[5],
]
pprint(controller.acq_light_configurations)
controller.set_lights(controller.acq_light_configurations[0], wait=1)
images_nwse, *_ = controller.acquire_leaf_disc(switch_state=True)

controller.acq_light_configurations = [
    LIGHTS_CYCLE[6],
    LIGHTS_CYCLE[7],
    LIGHTS_CYCLE[8],
    LIGHTS_CYCLE[9],
]
pprint(controller.acq_light_configurations)
controller.set_lights(controller.acq_light_configurations[0], wait=1)
images_lr, *_ = controller.acquire_leaf_disc(switch_state=True)

# controller.acq_light_configurations = [
#     LIGHTS_CYCLE[10],
#     LIGHTS_CYCLE[11],
#     LIGHTS_CYCLE[12],
#     LIGHTS_CYCLE[13],
# ]
# pprint(controller.acq_light_configurations)
# controller.set_lights(controller.acq_light_configurations[0], wait=1)
# images_one_off, *_ = controller.acquire_leaf_disc(switch_state=True)

accu, cx, cy, r = get_circles(images_full_light[0], color_space="hsv", channel="s")["accepted"][0]
crop_data = Rectangle.from_circle((cx,cy,r+16))

images_no_light = [crop_image(i, crop_data) for i in images_no_light]
images_full_light = [crop_image(i, crop_data) for i in images_full_light]
images_nwse = [crop_image(i, crop_data) for i in images_nwse]
images_lr = [crop_image(i, crop_data) for i in images_lr]
# images_one_off = [crop_image(i, crop_data) for i in images_one_off]

to_pil(
    concat_tile_resize(
        [
            [
                images_no_light[0],
                images_full_light[0],
                merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.MIN),
                #merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.MAX),
                #merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.AVG),
                #merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.MEDIAN),
            ],
            [
                images_no_light[0],
                images_full_light[0],
                merge_images(image_list=images_lr, merge_mode=ImageMergeMode.MIN),
                #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.MAX),
                #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.AVG),
                #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.MEDIAN),
            ],
            # [
            #     images_no_light[0],
            #     images_full_light[0],
            #     merge_images(image_list=images_one_off, merge_mode=ImageMergeMode.MIN),
            #     #merge_images(image_list=images_one_off, merge_mode=ImageMergeMode.MAX),
            #     #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.AVG),
            #     #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.MEDIAN),
            # ]
        ]
    )
)

In [ ]:
to_pil(images_full_light[0])

In [ ]:
to_pil(merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.MIN))

In [ ]:
to_pil(images_nwse[0])

In [ ]:
to_pil(concat_tile_resize([images_nwse, images_lr, images_one_off]))

In [ ]:
controller.move_relative(0,0,1)

In [ ]:
def get_ligths_checkboxes(index: int):
    cps = [
        cp.value
        for cp in [CardPoint.NORTH, CardPoint.WEST, CardPoint.SOUTH, CardPoint.EAST]
    ]
    return pn.widgets.CheckBoxGroup(
        name=f"{index}",
        options=cps,
        value=[v.value for v in LIGHTS_CYCLE[min(index, LEN_LIGHTS_CYCLE - 1)]],
        inline=True,
    )

In [ ]:
controller.acq_light_configurations = [
    LIGHTS_CYCLE[2],
    LIGHTS_CYCLE[3],
    LIGHTS_CYCLE[4],
    LIGHTS_CYCLE[5],
]
controller.acq_light_configurations

In [ ]:
col = pn.Column( get_ligths_checkboxes(1))
col

In [ ]:
controls = {
    "AeEnable": False,
    "ExposureTime": 5000,
    "AnalogueGain": 1,
    "AwbEnable": False,
    "ColourGains": (2.3, 0.9),
}

# controls = {
#     "AeEnable": True,
#     "AwbEnable": True,
# }


controller.camera.set_controls(controls)

In [ ]:
col.append(get_ligths_checkboxes(len(col)))
len(col)

In [ ]:
controller.cycle_lights()
controller.top_lights.mean

In [ ]:
controller.set_top_lights(
    True,
    card_points=[
        CardPoint.NORTH,
        CardPoint.SOUTH,
        CardPoint.EAST,
        CardPoint.WEST,
    ],
)

In [ ]:
img_east, _ = controller.capture_array()

In [ ]:
to_pil(img_east)

In [ ]:
to_pil(img_west)

In [ ]:
import numpy as np

image_grid([img_east, img_west,np.minimum(img_east,img_west)], row_count=2)

In [ ]:
image_data = controller.last_job_data

sel_image = pn.widgets.IntSlider(
    name="Select image",
    start=0,
    end=len(image_data) - 1,
    value=0,
    sizing_mode="stretch_width",
)
ph_image = pn.pane.Placeholder()
json_data = pn.pane.JSON()


@pn.depends(sel_image.param.value, watch=True)
def on_index_changed(index):
    image = image_data[index][0]
    metadata = image_data[index][1]
    ph_image.object = safe_pil_resize(to_pil(image), 600, 600)
    json_data.object = {k: str(v) for k, v in metadata.items() if k != "image"}


on_index_changed(sel_image.value)

pn.Column(sel_image, pn.Row(ph_image, json_data))